# Refined Stage 3E - paired significance tests

**No GPU.** Runs on whichever result files exist.

Every model forecasts the identical windows, so model differences are *paired* and the
Wilcoxon signed-rank test applies directly. Without this, a 4-7% gap between two models is
an anecdote.

This decides H2 — *the best model changes with granularity*. If the differences are not
significant, H2 does not survive and the thesis leans on H3 and H4 instead. Better to know
now than in the viva.

In [1]:
import sys, os, warnings, importlib
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import refined_stage_common as C

# A kernel that imported an older refined_stage_common keeps serving it from cache, and
# the failure surfaces much later as a missing column. Reload, then verify.
importlib.reload(C)
REQUIRED = "2026.09.10-relmae"
assert getattr(C, "__version__", None) == REQUIRED, (
    f"stale refined_stage_common (got {getattr(C, '__version__', 'none')}, "
    f"need {REQUIRED}) — restart the kernel and Run All")

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
PALETTE = {"chronos": "#2563eb", "timesfm": "#059669", "moirai": "#d97706"}
print(f"common module {C.__version__} from {C.__file__}")

common module 2026.09.10-relmae from e:\Thesis\EnergyForcastModel\refined_stage_common.py


In [2]:
FILES = {
    "stage2":  "refined_stage2_results.csv",
    "context": "refined_stage3b_context_sweep.csv",
    "horizon": "refined_stage3c_horizon_sweep.csv",
    "lead":    "refined_stage3d_fixed_lead_time.csv",
}
loaded = {}
for k, f in FILES.items():
    if os.path.exists(f):
        d_ = pd.read_csv(f)
        assert "relMAE" in d_.columns, (
            f"{f} predates relMAE — re-run the notebook that produced it")
        loaded[k] = C.clean(d_, verbose=False)
        print(f"{k:8s} {len(loaded[k]):>6,} rows  <- {f}")
    else:
        print(f"{k:8s} missing ({f})")
assert loaded, "no result files found — run at least refined_stage2_baseline first"

stage2    1,245 rows  <- refined_stage2_results.csv
context  18,675 rows  <- refined_stage3b_context_sweep.csv
horizon  18,162 rows  <- refined_stage3c_horizon_sweep.csv
lead     30,285 rows  <- refined_stage3d_fixed_lead_time.csv


## 1. Model vs model, within each cell

In [3]:
from scipy.stats import wilcoxon
from itertools import combinations

METRIC = "relMAE"      # switch to "MASE" to test the classic metric instead

def paired_models(df, cell=("etype", "gran_label"), metric=None):
    metric = metric or METRIC
    key = ["series_id", "origin_index", "context_steps", "horizon_steps"]
    out = []
    for cv, g in df.groupby(list(cell)):
        piv = g.pivot_table(index=key, columns="model", values=metric)
        for a, b in combinations(sorted(piv.columns), 2):
            pair = piv[[a, b]].dropna()
            if len(pair) < 6:
                continue
            try:
                st = wilcoxon(pair[a], pair[b])
                p = st.pvalue
            except ValueError:
                p = 1.0
            out.append({**dict(zip(cell, cv if isinstance(cv, tuple) else (cv,))),
                        "a": a, "b": b, "n": len(pair),
                        "median_a": pair[a].median(), "median_b": pair[b].median(),
                        "p": p, "significant": p < 0.05,
                        "winner": (a if pair[a].median() < pair[b].median() else b)
                                  if p < 0.05 else "—"})
    return pd.DataFrame(out)

sig = paired_models(loaded["stage2"])
print(sig.round(4).to_string(index=False))

etype       gran_label       a       b  n  median_a  median_b      p  significant  winner
 load            daily chronos  moirai 50    0.8369    0.8241 0.1032        False       —
 load            daily chronos timesfm 50    0.8369    0.4823 0.0000         True timesfm
 load            daily  moirai timesfm 50    0.8241    0.4823 0.0000         True timesfm
 load           hourly chronos  moirai 50    0.6829    1.0244 0.0000         True chronos
 load           hourly chronos timesfm 50    0.6829    0.9983 0.0069         True chronos
 load           hourly  moirai timesfm 50    1.0244    0.9983 0.0635        False       —
 load minutes (native) chronos  moirai 50    0.9645    1.1834 0.0001         True chronos
 load minutes (native) chronos timesfm 50    0.9645    0.7126 0.0635        False       —
 load minutes (native)  moirai timesfm 50    1.1834    0.7126 0.0000         True timesfm
solar            daily chronos  moirai 50    0.8023    0.8591 0.1515        False       —
solar     

In [4]:
n_sig = int(sig.significant.sum())
print(f"{n_sig} of {len(sig)} pairwise model comparisons are significant at p < 0.05.")
print()
print("Significant comparisons per cell:")
print(sig.groupby(["etype", "gran_label"])["significant"].agg(["sum", "size"]).to_string())
print()
if n_sig == 0:
    print("H2 DOES NOT SURVIVE on this evidence: no model difference is distinguishable")
    print("from noise. Reframe around H3 (cycle coverage) and H4 (zero-inflation).")
else:
    print("H2 has support where the comparisons are significant. Report only those cells")
    print("as model-selection findings; call the rest indistinguishable.")

13 of 27 pairwise model comparisons are significant at p < 0.05.

Significant comparisons per cell:
                        sum  size
etype gran_label                 
load  daily               2     3
      hourly              2     3
      minutes (native)    2     3
solar daily               0     3
      hourly              1     3
      minutes (native)    2     3
wind  daily               2     3
      hourly              1     3
      minutes (native)    1     3

H2 has support where the comparisons are significant. Report only those cells
as model-selection findings; call the rest indistinguishable.


### Multiple comparisons

Many tests are run here, so some will pass by chance. Holm-Bonferroni is applied below;
report the adjusted result, and say in the methodology that you did.

In [5]:
from statsmodels.stats.multitest import multipletests
if len(sig):
    rej, padj, _, _ = multipletests(sig["p"], method="holm")
    sig["p_holm"] = padj
    sig["significant_holm"] = rej
    print(f"After Holm correction: {int(rej.sum())} of {len(sig)} remain significant.")
    print(sig[sig.significant_holm][["etype", "gran_label", "a", "b",
                                     "median_a", "median_b", "p_holm", "winner"]]
             .round(4).to_string(index=False))
    sig.to_csv("refined_stage3e_model_significance.csv", index=False)

After Holm correction: 10 of 27 remain significant.
etype       gran_label       a       b  median_a  median_b  p_holm  winner
 load            daily chronos timesfm    0.8369    0.4823  0.0000 timesfm
 load            daily  moirai timesfm    0.8241    0.4823  0.0000 timesfm
 load           hourly chronos  moirai    0.6829    1.0244  0.0000 chronos
 load minutes (native) chronos  moirai    0.9645    1.1834  0.0033 chronos
 load minutes (native)  moirai timesfm    1.1834    0.7126  0.0000 timesfm
solar           hourly chronos timesfm    0.8642    0.9253  0.0002 chronos
solar minutes (native) chronos timesfm    2.4541    1.6004  0.0338 timesfm
solar minutes (native)  moirai timesfm    2.3616    1.6004  0.0251 timesfm
 wind            daily chronos timesfm    1.0304    0.9419  0.0040 timesfm
 wind minutes (native) chronos timesfm    0.8314    0.8070  0.0408 timesfm


## 2. Does more context help significantly? (H3)

In [6]:
if "context" in loaded:
    df = loaded["context"]
    out = []
    for (et, gk, mdl), g in df.groupby(["etype", "gran", "model"]):
        piv = g.pivot_table(index=["series_id", "origin_index"],
                            columns="context_steps", values=METRIC)
        ctxs = sorted(piv.columns)
        for a, b in zip(ctxs, ctxs[1:]):
            pair = piv[[a, b]].dropna()
            if len(pair) < 6:
                continue
            try:
                p = wilcoxon(pair[a], pair[b]).pvalue
            except ValueError:
                p = 1.0
            out.append({"etype": et, "gran": gk, "model": mdl, "from": a, "to": b,
                        "median_from": pair[a].median(), "median_to": pair[b].median(),
                        "improvement_%": 100*(1 - pair[b].median()/pair[a].median()),
                        "p": p, "significant": p < 0.05})
    ctx_sig = pd.DataFrame(out)
    print(ctx_sig.round(4).to_string(index=False))
    ctx_sig.to_csv("refined_stage3e_context_significance.csv", index=False)
else:
    print("run refined_stage3b_context_sweep first")

etype   gran   model  from   to  median_from  median_to  improvement_%      p  significant
 load     1D chronos   128  256       0.8719     0.8546         1.9798 0.3448        False
 load     1D chronos   256  512       0.8546     0.7748         9.3340 0.0000         True
 load     1D chronos   512 1024       0.7748     0.7508         3.1000 0.3346        False
 load     1D chronos  1024 2048       0.7508     0.7626        -1.5669 0.9992        False
 load     1D  moirai   128  256       1.0003     1.0004        -0.0101 0.6277        False
 load     1D  moirai   256  512       1.0004     0.9497         5.0689 0.0012         True
 load     1D  moirai   512 1024       0.9497     0.9053         4.6688 0.0004         True
 load     1D  moirai  1024 2048       0.9053     0.9314        -2.8764 0.5205        False
 load     1D timesfm   128  256       0.7286     0.6531        10.3636 0.0000         True
 load     1D timesfm   256  512       0.6531     0.4709        27.8924 0.0000         True

Next: `refined_stage3f_selection_matrix`.